# Сравнение стратегий `high_pr_auc` на задачах OpenML

Notebook сравнивает все 14 пресетов из `ml_toolkit/presets/classification/high_pr_auc/`
на реальных задачах бинарной классификации с дисбалансом классов из OpenML.

**Целевые метрики:** `PR-AUC`, `precision@K` (K = positive rate датасета), `ROC-AUC`.

**Режимы:**
- `QUICK_MODE = True` — 3 датасета, минимальные Optuna-триалы (~10–15 мин)
- `QUICK_MODE = False` — 20 датасетов, полные триалы (~4–5 ч)

**Логирование:** сырые скоры, best_params, таблицы → `experiments/high_pr_auc_benchmark/logs/`


In [ ]:
# Установка openml (нет в pyproject.toml, нужна только для этого notebook)
import importlib
import subprocess

if importlib.util.find_spec('openml') is None:
    subprocess.run(['uv', 'pip', 'install', 'openml'], check=True)

# Путь к корню проекта
from pathlib import Path
import sys

root = Path.cwd()
for _ in range(6):
    if (root / 'ml_toolkit' / '__init__.py').is_file():
        break
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print('project root:', root)
# Настройка логгера пакета
try:
    from ml_toolkit import configure_logging as _cfg_log
    _cfg_log()
except Exception:
    pass
if importlib.util.find_spec('openpyxl') is None:
    subprocess.run(['uv', 'pip', 'install', 'openpyxl'], check=True)


In [ ]:
import warnings

warnings.filterwarnings('ignore')

import datetime
import json
from pathlib import Path
import time
import traceback

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import openml
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder

from ml_toolkit.presets.classification.high_pr_auc import (
    AnomalyBlendClassifier,
    AsymmetricLossClassifier,
    BoostedEnsemble,
    CalibratedWrapper,
    EasyEnsembleClassifier,
    HardNegativeMiner,
    PrecisionAtKClassifier,
    PULearningClassifier,
    SelfTrainingBooster,
    SubsampleStacking,
    SyntheticOversamplingClassifier,
    ThresholdMovingCV,
    TwoStageCascade,
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110


In [ ]:
# ── Режим запуска ─────────────────────────────────────────────────────────────
QUICK_MODE = False   # False → ~4–5 ч на 20 датасетах с полными Optuna-триалами

N_DATASETS   = 3   if QUICK_MODE else 20   # датасетов для бенчмарка
N_OPTUNA     = 2   if QUICK_MODE else 10   # триалов Optuna на каждый пресет
CB_ITER_TUNE = 200 if QUICK_MODE else 400  # итераций CatBoost внутри триала
CB_ESR_TUNE  = 30  if QUICK_MODE else 50   # early stopping в триале
CB_ITER_FIT  = 300 if QUICK_MODE else 700  # итераций финального CatBoost
CB_ESR_FIT   = 40  if QUICK_MODE else 80   # early stopping финальной модели

print(f'Режим:         {"QUICK" if QUICK_MODE else "FULL (4–5 ч)"}'  )
print(f'Датасетов:     {N_DATASETS}')
print(f'Optuna/пресет: {N_OPTUNA}')


In [ ]:
# root уже определён в cell-01
EXPERIMENT_DIR = root / 'experiments' / 'high_pr_auc_benchmark'
LOGS_DIR       = EXPERIMENT_DIR / 'logs'
LOGS_DIR.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.datetime.now().strftime('%Y%m%d_%H%M')
RUN_DIR = LOGS_DIR / RUN_ID
RUN_DIR.mkdir(exist_ok=True)

print(f'Experiment : {EXPERIMENT_DIR}')
print(f'Logs dir   : {LOGS_DIR}')
print(f'Run ID     : {RUN_ID}')


In [ ]:
def precision_at_k_fraction(y_true: np.ndarray, scores: np.ndarray, k: float) -> float:
    """Доля позитивов в топ-k*N по score (k — дробь 0..1)."""
    n_top = max(1, int(len(y_true) * k))
    top_idx = np.argsort(-scores)[:n_top]
    return float(np.asarray(y_true)[top_idx].mean())

K_FRACTIONS = [0.01, 0.03, 0.05, 0.10]
K_LABELS    = ['P@1%', 'P@3%', 'P@5%', 'P@10%']


---
## 1. Поиск подходящих задач на OpenML

Критерии:
- Бинарная классификация, доля миноритарного класса **2–20%**
- Размер **3 000 – 200 000** строк, признаков **5–120**
- Нет пропущенных значений


In [ ]:
try:
    print('Скачиваем мета-информацию OpenML...')
    meta = openml.datasets.list_datasets(output_format='dataframe')
    print(f'Датасетов в реестре: {len(meta):,}')
    OPENML_META_AVAILABLE = True
except Exception as e:
    print(f'OpenML API недоступен ({type(e).__name__}): {str(e)[:80]}')
    meta = pd.DataFrame()
    OPENML_META_AVAILABLE = False


In [ ]:
# Кэшированные датасеты из предыдущих запусков
CACHED_DATASET_IDS = [43903, 44234, 46805, 47152]

if OPENML_META_AVAILABLE and len(meta) > 0:
    meta['minority_ratio'] = meta['MinorityClassSize'] / meta['NumberOfInstances']
    mask = (
        (meta['NumberOfClasses'] == 2) &
        (meta['NumberOfInstances'].between(3_000, 200_000)) &
        (meta['NumberOfFeatures'].between(5, 120)) &
        (meta['minority_ratio'].between(0.02, 0.20)) &
        (meta['NumberOfMissingValues'] == 0)
    )
    candidates = (
        meta[mask]
        .sort_values('NumberOfInstances')
        [['did', 'name', 'NumberOfInstances', 'NumberOfFeatures',
          'minority_ratio', 'NumberOfSymbolicFeatures']]
        .rename(columns={
            'did': 'dataset_id', 'NumberOfInstances': 'n_rows',
            'NumberOfFeatures': 'n_features', 'minority_ratio': 'pos_rate',
            'NumberOfSymbolicFeatures': 'n_categorical',
        })
        .reset_index(drop=True)
    )
    print(f'Подходящих датасетов: {len(candidates)}')
    display(candidates.head(10))
else:
    print('Используем закэшированные датасеты.')
    candidates = pd.DataFrame({'dataset_id': CACHED_DATASET_IDS})


In [ ]:
if OPENML_META_AVAILABLE and len(candidates) >= N_DATASETS and 'n_rows' in candidates.columns:
    # Равномерный выбор по диапазону размера (стратифицированный сэмпл)
    cands = candidates.sort_values('n_rows').reset_index(drop=True)
    step  = len(cands) / N_DATASETS
    idxs  = sorted({int(i * step + step / 2) for i in range(N_DATASETS)})
    idxs  = [min(i, len(cands) - 1) for i in idxs]
    selected = cands.iloc[idxs].copy()
    print('Выбранные датасеты (стратифицированный сэмпл по размеру):')
    display(selected[['dataset_id', 'name', 'n_rows', 'pos_rate', 'n_categorical']])
    SELECTED_DATASET_IDS = selected['dataset_id'].astype(int).tolist()
else:
    SELECTED_DATASET_IDS = CACHED_DATASET_IDS[:N_DATASETS]
    print('Используем закэшированные IDs:', SELECTED_DATASET_IDS)

print(f'\nИтого датасетов: {len(SELECTED_DATASET_IDS)}')


---
## 2. Загрузка и подготовка данных

In [ ]:
def load_openml_dataset(did: int) -> dict:
    ds = openml.datasets.get_dataset(
        did, download_data=True,
        download_qualities=False, download_features_meta_data=False,
    )
    X, y, _, _ = ds.get_data(
        target=ds.default_target_attribute, dataset_format='dataframe',
    )
    if ds.default_target_attribute in X.columns:
        X = X.drop(columns=[ds.default_target_attribute])

    le = LabelEncoder()
    y_enc = pd.Series(le.fit_transform(y.astype(str)), name='target')
    if y_enc.mean() > 0.5:
        y_enc = 1 - y_enc

    # Детектируем категориальные по dtype (cat_indicator у OpenML ненадёжен)
    cat_features = X.select_dtypes(include=['category', 'object', 'string']).columns.tolist()
    for col in cat_features:
        X[col] = X[col].astype(str).fillna('__NA__')
    for col in [c for c in X.columns if c not in cat_features]:
        if X[col].isna().any():
            X[col] = X[col].fillna(X[col].median())

    return dict(
        name=ds.name, did=did, X=X, y=y_enc,
        cat_features=cat_features,
        pos_rate=float(y_enc.mean()),
        n_rows=len(X), n_features=X.shape[1],
    )


def prepare_splits(info: dict, valid_size: float = 0.20,
                   test_size: float = 0.20, random_state: int = 42) -> dict:
    X, y = info['X'], info['y']
    X_tv, X_te, y_tv, y_te = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=random_state)
    X_tr, X_va, y_tr, y_va = train_test_split(
        X_tv, y_tv, test_size=valid_size / (1 - test_size),
        stratify=y_tv, random_state=random_state)
    return {
        **info,
        'X_train': X_tr.reset_index(drop=True),
        'X_valid': X_va.reset_index(drop=True),
        'X_test':  X_te.reset_index(drop=True),
        'y_train': y_tr.reset_index(drop=True),
        'y_valid': y_va.reset_index(drop=True),
        'y_test':  y_te.reset_index(drop=True),
    }


In [ ]:
DATASETS = {}
for did in SELECTED_DATASET_IDS:
    try:
        info = load_openml_dataset(did)
        data = prepare_splits(info)
        DATASETS[info['name']] = data
        print(
            f"✓ [{did:6d}] {info['name']:35s} "
            f"n={len(data['X_train'])+len(data['X_valid']):>7,} "
            f"pos={data['y_train'].mean():.3f} "
            f"cat={len(data['cat_features'])}",
        )
    except Exception as e:
        print(f'✗ [{did}] {e}')

print(f'\nЗагружено датасетов: {len(DATASETS)}')


### Характеристики выбранных датасетов

In [ ]:
rows = []
for name, d in DATASETS.items():
    rows.append({
        'Датасет':    name,
        'N train':    len(d['X_train']),
        'N valid':    len(d['X_valid']),
        'N test':     len(d['X_test']),
        'N features': d['n_features'],
        'N cat':      len(d['cat_features']),
        'pos% train': f"{d['y_train'].mean()*100:.2f}%",
        'Pos:Neg':    f"1:{int(round(1/d['y_train'].mean()-1))}",
        'k (P@k)':    f"{d['pos_rate']*100:.1f}%",
    })
pd.DataFrame(rows).set_index('Датасет')


---
## 3. Бенчмарк

### 3.1 Конфигурации пресетов

`k_fraction` в `PrecisionAtKClassifier` = `pos_rate` датасета (оптимизируем по реальному числу позитивов).

**Пресеты с Optuna:** `PrecisionAtK`, `TwoStageCascade`, `HardNegativeMiner`, `PULearning`, `AsymmetricLoss`.
Остальные используют внутреннюю оптимизацию (ансамбли, grid-поиск по α, CV-порог).


In [ ]:
# Пресеты, требующие числовых признаков (без нативной поддержки str cat)
NUMERIC_ONLY_PRESETS = {'SMOTE', 'EasyEnsemble', 'Calibrated(Easy)', 'AnomalyBlend'}


def _base_cb(n_iter=None, esr=None):
    """Параметры CatBoost для прямого (не-Optuna) fit."""
    return {
        'iterations':           n_iter or CB_ITER_FIT,
        'max_depth':            6,
        'learning_rate':        0.05,
        'l2_leaf_reg':          3.0,
        'subsample':            0.8,
        'min_data_in_leaf':     5,
        'early_stopping_rounds': esr or CB_ESR_FIT,
        'random_seed':          42,
        'verbose':              0,
    }


def make_preset_factories(data: dict) -> dict:
    """Возвращает фабрики пресетов, адаптированные под конкретный датасет.

    k_fraction для PrecisionAtKClassifier = pos_rate датасета,
    чтобы метрика «precision в топ-k» соответствовала реальному
    числу позитивов (precision при recall → 1 в идеале).
    """
    k_frac = round(data['pos_rate'], 4)

    return {
        # ── Ансамблевые (без Optuna: внутренняя оптимизация) ────────────────
        'BoostedEnsemble': lambda: BoostedEnsemble(
            averaging='rank',
            base_params=_base_cb(),
        ),
        'EasyEnsemble': lambda: EasyEnsembleClassifier(
            n_estimators=8 if QUICK_MODE else 20,
            neg_ratio=10,
            base='lightgbm',
        ),
        'SubsampleStacking': lambda: SubsampleStacking(
            n_base_models=3 if QUICK_MODE else 7,
            subsample_rate=0.75,
            meta='logistic',
        ),
        'SMOTE': lambda: SyntheticOversamplingClassifier(
            method='smote',
            sampling_strategy='auto',
            base='lightgbm',
        ),
        'SelfTraining': lambda: SelfTrainingBooster(
            n_rounds=2 if QUICK_MODE else 4,
            pseudo_weight=0.3,
            max_pseudo_ratio=2.0,
            base_params=_base_cb(),
        ),
        'AnomalyBlend': lambda: AnomalyBlendClassifier(
            n_if_estimators=50 if QUICK_MODE else 150,
            n_alpha_steps=21 if QUICK_MODE else 51,
            supervised_params=_base_cb(),
        ),

        # ── С Optuna ────────────────────────────────────────────────────────
        # k_fraction = pos_rate: оптимизируем «precision при отборе ровно
        # столько объектов, сколько реальных позитивов»
        'PrecisionAtK(pos_rate)': lambda: PrecisionAtKClassifier(
            k_fraction=k_frac,
            n_optuna_trials=N_OPTUNA,
            calibrate=True,
        ),
        'TwoStageCascade': lambda: TwoStageCascade(
            recall_target=0.90,
            stage1_n_trials=N_OPTUNA,
            stage2_n_trials=N_OPTUNA,
        ),
        'HardNegativeMiner': lambda: HardNegativeMiner(
            n_rounds=2 if QUICK_MODE else 4,
            hard_percentile=0.80,
            hard_weight=4.0,
            n_optuna_trials=N_OPTUNA,
        ),
        'PULearning': lambda: PULearningClassifier(
            n_optuna_trials=N_OPTUNA,
        ),
        'AsymmetricLoss': lambda: AsymmetricLossClassifier(
            gamma_pos=0.0,
            gamma_neg=4.0,
            prob_margin=0.05,
            n_optuna_trials=N_OPTUNA,
        ),

        # ── Обёртки ─────────────────────────────────────────────────────────
        'Calibrated(Easy)': lambda: CalibratedWrapper(
            EasyEnsembleClassifier(
                n_estimators=6 if QUICK_MODE else 15,
                neg_ratio=10,
            ),
            method='isotonic',
        ),
        'Threshold+Cascade': lambda: ThresholdMovingCV(
            TwoStageCascade(recall_target=0.90, stage1_n_trials=N_OPTUNA),
            optimize='f2',
        ),
    }


print(f'Пресетов в шаблоне: {len(make_preset_factories(list(DATASETS.values())[0]))}')


In [ ]:
def _encode_cats_for_numeric(X: pd.DataFrame, cat_features: list) -> pd.DataFrame:
    if not cat_features:
        return X
    X = X.copy()
    enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X[cat_features] = enc.fit_transform(X[cat_features])
    return X


def compute_metrics(y_true: pd.Series, scores: np.ndarray,
                    prefix: str, k_frac: float) -> dict:
    """PR-AUC, ROC-AUC, precision@k для заданного сплита."""
    y = y_true.to_numpy()
    res = {
        f'pr_auc_{prefix}':  round(average_precision_score(y, scores), 4),
        f'roc_auc_{prefix}': round(roc_auc_score(y, scores), 4),
        # Адаптивный P@k (k = pos_rate датасета)
        f'P@posrate_{prefix}': round(precision_at_k_fraction(y, scores, k_frac), 4),
    }
    for k, lbl in zip(K_FRACTIONS, K_LABELS):
        res[f'{lbl}_{prefix}'] = round(precision_at_k_fraction(y, scores, k), 4)
    return res


def _save_run_artifacts(preset_name: str, ds_name: str, run_dir: Path,
                        val_scores, test_scores, best_params, fit_time):
    """Сохраняет сырые артефакты прогона в logs/{RUN_ID}/{ds}/{preset}/."""
    pdir = run_dir / ds_name.replace('/', '_') / preset_name.replace('/', '_')
    pdir.mkdir(parents=True, exist_ok=True)
    np.save(pdir / 'val_scores.npy', val_scores.astype(np.float32))
    np.save(pdir / 'test_scores.npy', test_scores.astype(np.float32))
    meta = {'preset': preset_name, 'dataset': ds_name, 'fit_time_s': fit_time}
    if best_params:
        meta['best_params'] = {k: (float(v) if isinstance(v, (int, float)) else str(v))
                               for k, v in best_params.items()}
    with open(pdir / 'meta.json', 'w') as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)


def run_preset(preset_name: str, preset, data: dict, run_dir: Path) -> dict:
    X_tr, y_tr = data['X_train'], data['y_train']
    X_va, y_va = data['X_valid'], data['y_valid']
    X_te, y_te = data['X_test'],  data['y_test']
    feats      = list(X_tr.columns)
    cat_feats  = data['cat_features']
    k_frac     = data['pos_rate']

    if preset_name in NUMERIC_ONLY_PRESETS and cat_feats:
        X_tr = _encode_cats_for_numeric(X_tr, cat_feats)
        X_va = _encode_cats_for_numeric(X_va, cat_feats)
        X_te = _encode_cats_for_numeric(X_te, cat_feats)
        eff_cats = []
    else:
        eff_cats = cat_feats

    row = {'preset': preset_name, 'dataset': data['name'],
           'k_frac': round(k_frac, 4), 'status': 'OK'}
    try:
        t0 = time.time()
        preset.fit(X_tr, y_tr, X_va, y_va,
                   selected_features=feats, cat_features=eff_cats)
        row['fit_time_s'] = round(time.time() - t0, 1)

        val_scores  = np.asarray(preset.valid_pred_).ravel()
        test_scores = np.asarray(preset.predict_proba(X_te[feats])).ravel()

        row.update(compute_metrics(y_va, val_scores,  'val',  k_frac))
        row.update(compute_metrics(y_te, test_scores, 'test', k_frac))

        _save_run_artifacts(
            preset_name, data['name'], run_dir,
            val_scores, test_scores,
            getattr(preset, 'best_params_', None),
            row['fit_time_s'],
        )
    except Exception:
        row['status']     = traceback.format_exc().split('\n')[-2][:120]
        row['fit_time_s'] = None

    return row


def run_benchmark(datasets: dict, run_dir: Path) -> pd.DataFrame:
    results   = []
    n_total   = sum(len(make_preset_factories(d)) for d in datasets.values())
    done      = 0

    for ds_name, data in datasets.items():
        factories = make_preset_factories(data)
        print(f'\n── {ds_name}  '
              f'(n={len(data["X_train"]):,}  pos={data["y_train"].mean():.3f}  '
              f'k={data["pos_rate"]:.3f})')

        for preset_name, factory in factories.items():
            done += 1
            print(f'  [{done:>3}/{n_total}] {preset_name:<30}', end=' ', flush=True)

            try:
                preset = factory()
            except Exception:
                msg = traceback.format_exc().split('\n')[-2][:80]
                print(f'init ERROR: {msg}')
                results.append({'preset': preset_name, 'dataset': ds_name,
                                'status': f'init_error: {msg}'})
                continue

            row = run_preset(preset_name, preset, data, run_dir)

            if row['status'] == 'OK':
                print(
                    f"PR-AUC={row['pr_auc_val']:.4f}  "
                    f"P@pos={row.get('P@posrate_val', float('nan')):.4f}  "
                    f"t={row['fit_time_s']}s"
                )
            else:
                print(f'ERROR: {row["status"][:80]}')

            results.append(row)

    return pd.DataFrame(results)


---
## 4. Запуск бенчмарка

> **QUICK_MODE:** ~10–15 мин  |  **FULL_MODE:** ~4–5 ч


In [ ]:
t_start    = time.time()
results_df = run_benchmark(DATASETS, RUN_DIR)
elapsed    = (time.time() - t_start) / 60
print(f'\n✓ Завершено за {elapsed:.1f} мин')

ok  = results_df[results_df['status'] == 'OK']
err = results_df[results_df['status'] != 'OK']
print(f'OK: {len(ok)}  Ошибок: {len(err)}')
if not err.empty:
    display(err[['preset', 'dataset', 'status']])


In [ ]:
# Нормализуем имена датасетов: lower, дефисы→подчёркивания, дедупликация
def _norm_ds(name: str) -> str:
    return name.lower().replace('-', '_').replace(' ', '_')

results_df['dataset'] = results_df['dataset'].map(_norm_ds)
results_df = results_df.drop_duplicates(subset=['preset', 'dataset'], keep='first')

# Сохраняем
_to_xlsx(results_df, EXPERIMENT_DIR / 'benchmark_results.xlsx')
_to_xlsx(results_df, RUN_DIR / 'results.xlsx')

ok  = results_df[results_df['status'] == 'OK'].copy()
err = results_df[results_df['status'] != 'OK'].copy()

print(f'Датасетов: {ok["dataset"].nunique()}   Пресетов: {ok["preset"].nunique()}   OK-строк: {len(ok)}')
print('Датасеты:', sorted(ok['dataset'].unique()))


---
## 5. Анализ результатов

In [ ]:
from openpyxl.utils import get_column_letter as _gcl


def _to_xlsx(df: pd.DataFrame, path, index: bool = False, max_col_width: int = 60):
    """Сохраняет DataFrame в xlsx с оптимальной шириной колонок."""
    path = str(path).replace('.csv', '') + '.xlsx'
    _df = df.reset_index() if index else df
    with pd.ExcelWriter(path, engine='openpyxl') as writer:
        _df.to_excel(writer, index=False, sheet_name='Sheet1')
        ws = writer.sheets['Sheet1']
        for col_idx, col_name in enumerate(_df.columns, start=1):
            header_len = len(str(col_name))
            vals = _df[col_name].astype(str)
            data_len = int(vals.str.len().quantile(0.95)) if len(vals) else 0
            ws.column_dimensions[_gcl(col_idx)].width = min(
                max(header_len, data_len) + 2, max_col_width
            )

import matplotlib.patches as mpatches

PRESET_ORDER = (
    ok.groupby('preset')['pr_auc_test'].mean()
      .sort_values(ascending=False).index.tolist()
)

PALETTE = [
    '#2563EB','#16A34A','#DC2626','#D97706','#7C3AED',
    '#0891B2','#BE185D','#65A30D','#EA580C','#6366F1',
    '#14B8A6','#F59E0B','#EF4444',
]
COLOR_MAP = {p: PALETTE[i % len(PALETTE)] for i, p in enumerate(PRESET_ORDER)}

def _save_fig(name, dpi_file=130):
    path = EXPERIMENT_DIR / f'{name}.png'
    plt.savefig(path, dpi=dpi_file, bbox_inches='tight')
    plt.savefig(RUN_DIR / f'{name}.png', dpi=100, bbox_inches='tight')


def rank_presets(ok_df, metric):
    if metric not in ok_df.columns:
        return pd.DataFrame()
    ranked = (
        ok_df.dropna(subset=[metric])
             .assign(rank=lambda d: d.groupby('dataset')[metric]
                     .rank(ascending=False, method='min'))
    )
    return (
        ranked.groupby('preset')
              .agg(mean_rank=('rank','mean'), best_rank=('rank','min'),
                   wins=('rank', lambda x: (x==1).sum()),
                   mean_score=(metric,'mean'), std_score=(metric,'std'))
              .sort_values('mean_rank').round(4).reset_index()
    )


def _heatmap(ok_df, metric, title, fname):
    """Тепловая карта с выделением лучшего значения в каждой колонке."""
    pivot = ok_df.pivot_table(index='preset', columns='dataset', values=metric)
    pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]

    # Маска: лучший в каждой колонке
    best_mask = pivot == pivot.max(axis=0)

    fig, ax = plt.subplots(figsize=(max(9, len(pivot.columns) * 1.9),
                                    len(pivot) * 0.6 + 1.2))
    sns.heatmap(pivot, ax=ax, annot=True, fmt='.3f',
                cmap='RdYlGn', linewidths=0.4, linecolor='#888',
                cbar_kws={'label': metric, 'shrink': 0.7})

    # Обводим лучшую ячейку в колонке жирной рамкой
    for col_idx in range(pivot.shape[1]):
        for row_idx in range(pivot.shape[0]):
            if best_mask.iloc[row_idx, col_idx]:
                ax.add_patch(mpatches.FancyBboxPatch(
                    (col_idx, row_idx), 1, 1,
                    boxstyle='square,pad=0', linewidth=2.5,
                    edgecolor='#1e3a8a', facecolor='none',
                    transform=ax.transData, zorder=5))

    ax.set_title(title, fontsize=13, pad=12, fontweight='bold')
    ax.set_xlabel(''); ax.set_ylabel('')
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    ax.tick_params(axis='y', rotation=0,  labelsize=8)
    plt.tight_layout()
    _save_fig(fname)
    plt.show()


### 5.1 PR-AUC — тепловые карты (val и test)

In [ ]:
_heatmap(ok, 'pr_auc_val',  'PR-AUC (validation)', 'pr_auc_heatmap_val')
_heatmap(ok, 'pr_auc_test', 'PR-AUC (test)',        'pr_auc_heatmap_test')


### 5.2 Precision@posrate (val и test)

In [ ]:
for split in ('val', 'test'):
    col = f'P@posrate_{split}'
    if col in ok.columns:
        _heatmap(ok, col,
                 f'Precision@K  (K = pos_rate датасета, {split})',
                 f'precision_at_posrate_heatmap_{split}')


### 5.3 Val → Test: разница в score

In [ ]:
delta_metrics = [('pr_auc_val', 'pr_auc_test'), ('P@posrate_val', 'P@posrate_test')]

for val_col, test_col in delta_metrics:
    if val_col not in ok.columns or test_col not in ok.columns:
        continue
    delta = ok.copy()
    delta['delta'] = delta[test_col] - delta[val_col]

    pivot_delta = delta.pivot_table(index='preset', columns='dataset', values='delta')
    pivot_delta = pivot_delta.loc[
        delta.groupby('preset')['delta'].mean().sort_values(ascending=False).index
    ]

    fig, ax = plt.subplots(figsize=(max(9, len(pivot_delta.columns) * 1.9),
                                    len(pivot_delta) * 0.6 + 1.2))
    vmax = pivot_delta.abs().max().max()
    sns.heatmap(pivot_delta, ax=ax, annot=True, fmt='+.3f',
                cmap='RdBu', center=0, vmin=-vmax, vmax=vmax,
                linewidths=0.4, linecolor='#888',
                cbar_kws={'label': 'test − val', 'shrink': 0.7})
    ax.set_title(f'Разница test − val  [{val_col.split("_")[0]}_{val_col.split("_")[1]}]',
                 fontsize=13, pad=12, fontweight='bold')
    ax.set_xlabel(''); ax.set_ylabel('')
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    ax.tick_params(axis='y', rotation=0,  labelsize=8)
    plt.tight_layout()
    _save_fig(f'delta_{val_col.replace("@","at")}')
    plt.show()

    # Средняя дельта по пресетам
    mean_delta = delta.groupby('preset')['delta'].mean().sort_values(ascending=False)
    print(f'\n=== Средняя delta (test−val) по {val_col} ===')
    display(mean_delta.round(4).to_frame('mean_delta'))


### 5.4 Рейтинговые таблицы

In [ ]:
for split in ('val', 'test'):
    for base in ('pr_auc', 'P@posrate'):
        col = f'{base}_{split}'
        r = rank_presets(ok, col)
        if not r.empty:
            print(f'=== {col} ===')
            display(r)
            _to_xlsx(r, RUN_DIR / f'ranking_{base}_{split}.xlsx')


### 5.5 Win-rate и время обучения

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, max(4, ok['preset'].nunique() * 0.42 + 1)))

for ax, metric, label in [
    (axes[0], 'pr_auc_val',  'PR-AUC val'),
    (axes[1], 'pr_auc_test', 'PR-AUC test'),
]:
    best = ok.loc[ok.groupby('dataset')[metric].idxmax(), 'preset']
    vc = best.value_counts().reset_index()
    vc.columns = ['preset', 'wins']
    colors = [COLOR_MAP.get(p, '#999') for p in vc['preset']]
    if not vc.empty:
        bars = ax.barh(vc['preset'], vc['wins'], color=colors, edgecolor='white', height=0.65)
        ax.bar_label(bars, padding=3, fontsize=9)
    ax.set_xlabel(f'Побед ({label})')
    ax.set_title(f'Win-rate  ({label})', fontweight='bold')
    ax.spines[['top','right']].set_visible(False)

time_df = (
    ok.groupby('preset')['fit_time_s'].mean()
      .sort_values().reset_index()
      .rename(columns={'fit_time_s': 'mean_fit_time_s'}).round(1)
)
colors_t = [COLOR_MAP.get(p, '#999') for p in time_df['preset']]
bars_t = axes[2].barh(time_df['preset'], time_df['mean_fit_time_s'],
                      color=colors_t, edgecolor='white', height=0.65)
axes[2].bar_label(bars_t, fmt='%.0f s', padding=3, fontsize=9)
axes[2].set_xlabel('Среднее время fit (с)')
axes[2].set_title('Время обучения', fontweight='bold')
axes[2].spines[['top','right']].set_visible(False)

plt.tight_layout()
_save_fig('win_rate_and_time')
plt.show()

_to_xlsx(time_df, RUN_DIR / 'fit_times.xlsx')


### 5.6 Precision@K кривые: влияние размера выборки на точность

In [ ]:
K_COLS  = ['P@1%_test', 'P@3%_test', 'P@5%_test', 'P@10%_test']
K_VALS  = [0.01, 0.03, 0.05, 0.10]
avail_k = [(c, v) for c, v in zip(K_COLS, K_VALS) if c in ok.columns]

datasets_list = sorted(ok['dataset'].unique())
n_ds  = len(datasets_list)
ncols = min(3, n_ds)
nrows = -(-n_ds // ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(7.5 * ncols, 4.5 * nrows), squeeze=False)

for idx, ds in enumerate(datasets_list):
    ax = axes[idx // ncols][idx % ncols]
    sub = ok[ok['dataset'] == ds]
    pos_rate = sub['k_frac'].iloc[0] if 'k_frac' in sub.columns else None

    for _, row in sub.sort_values('pr_auc_test', ascending=False).iterrows():
        preset = row['preset']
        xs = [v for c, v in avail_k if not pd.isna(row.get(c, float('nan')))]
        ys = [row[c] for c, v in avail_k if not pd.isna(row.get(c, float('nan')))]
        if not ys:
            continue
        ax.plot(xs, ys, marker='o', markersize=4, linewidth=1.6,
                color=COLOR_MAP.get(preset, '#999'), label=preset, alpha=0.85)
        if 'P@posrate_test' in row and not pd.isna(row['P@posrate_test']) and pos_rate:
            ax.plot(pos_rate, row['P@posrate_test'], marker='*', markersize=9,
                    color=COLOR_MAP.get(preset, '#999'), zorder=5)

    if pos_rate:
        ax.axvline(pos_rate, color='#666', linewidth=0.9, linestyle='--', alpha=0.6)
        ax.text(pos_rate, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 1.0,
                f' pos={pos_rate:.1%}', fontsize=7, color='#555', va='top')

    n_pos = sub['k_frac'].iloc[0] * len(sub) if 'k_frac' in sub.columns else 0
    ax.set_title(f'{ds}', fontsize=9, fontweight='bold')
    ax.set_xlabel('Доля отобранных объектов (K)', fontsize=8)
    ax.set_ylabel('Precision@K (test)', fontsize=8)
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0, decimals=0))
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.25, linestyle=':')
    ax.spines[['top','right']].set_visible(False)

for idx in range(n_ds, nrows * ncols):
    axes[idx // ncols][idx % ncols].axis('off')

handles = [plt.Line2D([0],[0], color=COLOR_MAP.get(p,'#999'), marker='o',
                      markersize=4, linewidth=1.6, label=p)
           for p in PRESET_ORDER if p in COLOR_MAP]
fig.legend(handles=handles, loc='lower center', ncol=min(7, len(PRESET_ORDER)),
           fontsize=8, bbox_to_anchor=(0.5, -0.02), framealpha=0.95)
plt.suptitle('Precision@K кривые по датасетам (test)\n★ = K=pos_rate', fontsize=13, y=1.01)
plt.tight_layout()
_save_fig('precision_k_curves_by_dataset')
plt.show()


### 5.7 Визуализации для публикации

In [ ]:
# ── Plot 1: Рейтинг пресетов по PR-AUC test (горизонтальный barplot) ──────
r_test = rank_presets(ok, 'pr_auc_test').set_index('preset')

fig, ax = plt.subplots(figsize=(8, 6))
presets_sorted = r_test.sort_values('mean_score')['mean_score']
colors = [COLOR_MAP.get(p, '#999') for p in presets_sorted.index]
bars = ax.barh(presets_sorted.index, presets_sorted.values,
               color=colors, edgecolor='white', height=0.7)

# Добавляем std как errorbar
stds = [r_test.loc[p, 'std_score'] for p in presets_sorted.index]
ax.errorbar(presets_sorted.values, range(len(presets_sorted)),
            xerr=stds, fmt='none', color='#333', linewidth=1.2,
            capsize=3, capthick=1.2)

ax.bar_label(bars, fmt='%.3f', padding=5, fontsize=9)
ax.set_xlabel('Mean PR-AUC (test)', fontsize=11)
ax.set_title('Рейтинг стратегий по PR-AUC на тестовой выборке\n'
             '(среднее ± σ по датасетам)', fontsize=12, fontweight='bold', pad=12)
ax.set_xlim(0, presets_sorted.max() * 1.18)
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='x', alpha=0.25, linestyle=':')
plt.tight_layout()
_save_fig('pub_ranking_pr_auc_test', dpi_file=150)
plt.show()


# ── Plot 2: Val vs Test scatter ─────────────────────────────────────────────
val_m  = ok.groupby('preset')['pr_auc_val'].mean()
test_m = ok.groupby('preset')['pr_auc_test'].mean()
delta  = (test_m - val_m).sort_values()

fig, ax = plt.subplots(figsize=(7, 6))
ax.axhline(0, color='#aaa', linewidth=0.9, linestyle='--')
colors_d = [COLOR_MAP.get(p,'#999') for p in delta.index]
bars = ax.barh(delta.index, delta.values, color=colors_d,
               edgecolor='white', height=0.7)
ax.bar_label(bars, fmt='%+.3f', padding=4, fontsize=9)
ax.set_xlabel('PR-AUC test − PR-AUC val', fontsize=11)
ax.set_title('Generalization gap: test − val\n(+ = лучше обобщается)',
             fontsize=12, fontweight='bold', pad=12)
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='x', alpha=0.25, linestyle=':')
plt.tight_layout()
_save_fig('pub_generalization_gap', dpi_file=150)
plt.show()


# ── Plot 3: Win-rate bubble chart ───────────────────────────────────────────
r_df = rank_presets(ok, 'pr_auc_test').set_index('preset')
fig, ax = plt.subplots(figsize=(8, 6))
for preset, row in r_df.iterrows():
    ax.scatter(row['mean_score'], row['mean_rank'],
               s=max(row['wins'], 0.3) * 400 + 50,
               color=COLOR_MAP.get(preset,'#999'),
               alpha=0.85, edgecolors='white', linewidth=1.5, zorder=3)
    ax.annotate(preset, (row['mean_score'], row['mean_rank']),
                textcoords='offset points', xytext=(8, 0),
                fontsize=8, va='center')

ax.invert_yaxis()
ax.set_xlabel('Mean PR-AUC test', fontsize=11)
ax.set_ylabel('Средний ранг (↑ лучше)', fontsize=11)
ax.set_title('Стратегии: score vs ранг\n(размер пузыря = число побед)',
             fontsize=12, fontweight='bold', pad=12)
ax.spines[['top','right']].set_visible(False)
ax.grid(alpha=0.2, linestyle=':')
plt.tight_layout()
_save_fig('pub_score_vs_rank_bubble', dpi_file=150)
plt.show()


### 5.8 Лучшие параметры Optuna

In [ ]:
optuna_presets = ['AsymmetricLoss', 'PULearning', 'HardNegativeMiner',
                  'TwoStageCascade', 'PrecisionAtK(pos_rate)']
rows_bp = []
for ds_dir in sorted(RUN_DIR.iterdir()):
    if not ds_dir.is_dir():
        continue
    for pr_dir in sorted(ds_dir.iterdir()):
        meta_path = pr_dir / 'meta.json'
        if not meta_path.exists():
            continue
        with open(meta_path) as f:
            meta = json.load(f)
        if meta.get('preset') not in optuna_presets:
            continue
        bp = meta.get('best_params', {})
        bp['dataset'] = meta.get('dataset', ds_dir.name)
        bp['preset']  = meta.get('preset', pr_dir.name)
        rows_bp.append(bp)

if rows_bp:
    bp_df = pd.DataFrame(rows_bp).set_index(['dataset','preset'])
    print('Лучшие параметры Optuna:')
    display(bp_df)
    _to_xlsx(bp_df, RUN_DIR / 'best_params.xlsx', index=True)
else:
    print('Нет сохранённых параметров Optuna')


---
## Выводы

### Как читать результаты

| Паттерн | Вероятная интерпретация |
|---------|------------------------|
| Высокий val, низкий test | Overfitting через Optuna на val-сплит |
| Низкий val, высокий test | Метод хорошо обобщается без переподгонки |
| Красная ячейка в delta-карте | Метод деградирует на тесте |
| Пустая ячейка | Пресет упал с ошибкой на этом датасете |
